# ZS601 colored LiDAR initialization: 200 virtual views

This notebook executes zero optimization steps. It saves and reloads an SH0 Gaussian PLY
with opacity 0.999999, renders all fixed near cameras, and evaluates against Blender GT.
Run through the official Colab CLI. Upload the input bundle and source archive first.
The executed notebook, terminal outputs, and hashes are retained. Existing outputs are refused.

See the adjacent README for coordinate, PNG, alpha, scale and license contracts.


In [1]:
from pathlib import Path
import os, sys, subprocess, json, hashlib, time, platform
import torch
ROOT = Path(os.environ.get('ZS601_RUN_ROOT', '/content/zs601-lidar-init-v001'))
REPO = ROOT/'source'
PKG = REPO/'gaussian-splatting-lidar-init'
INPUT = ROOT/'input'
RUN = ROOT/os.environ.get('ZS601_ATTEMPT', 'run-001')
RUN.mkdir(exist_ok=False)
def sha(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1024*1024),b''): h.update(b)
    return h.hexdigest()
def save(name,data):
    with (RUN/name).open('x') as f: json.dump(data,f,indent=2,allow_nan=False)
def run(command, log):
    print('$', ' '.join(map(str,command)), flush=True)
    with (RUN/log).open('x') as f:
        proc=subprocess.Popen(list(map(str,command)),cwd=PKG,env=os.environ.copy(),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in proc.stdout:
            f.write(line); f.flush(); print(line,end='',flush=True)
        rc=proc.wait()
    if rc: raise RuntimeError(f'{log} failed: exit {rc}')
env=dict(python=platform.python_version(),torch=torch.__version__,cuda=torch.version.cuda,
         gpu=torch.cuda.get_device_name(0),capability=torch.cuda.get_device_capability(0),
         source_commit=os.environ.get('ZS601_CODE_COMMIT'),optimization_steps=0)
save('environment.json',env)
print(json.dumps(env,indent=2))
assert torch.cuda.is_available()
assert env['source_commit'], 'Supply ZS601_CODE_COMMIT from the verified source archive manifest.'


{
  "python": "3.13.15",
  "torch": "2.11.0+cu128",
  "cuda": "12.8",
  "gpu": "NVIDIA L4",
  "capability": [
    8,
    9
  ],
  "source_commit": "4c7186e363f14050c4977bb192f12ed237112764",
  "optimization_steps": 0
}


In [2]:
# Verify immutable source and every input before CUDA build or rendering.
source_manifest=json.loads((ROOT/'source_manifest.json').read_text())
for record in source_manifest['files']:
    assert sha(REPO/record['path']) == record['sha256'], record['path']
assert source_manifest['code_commit'] == env['source_commit']
input_manifest=json.loads((INPUT/'input_manifest.json').read_text())
for record in input_manifest['files']:
    assert sha(INPUT/record['path']) == record['sha256'], record['path']
save('identity_verified.json',dict(code_commit=env['source_commit'],source_files=len(source_manifest['files']),
    input_files=len(input_manifest['files']),point_cloud_sha256=sha(INPUT/'points_3cm.ply')))
print('All source and input hashes verified.')


All source and input hashes verified.


In [3]:
# Use the runtime's PyTorch/CUDA; compile pinned CUDA sources with the explicit cstdint include.
os.environ['MAX_JOBS']='2'
os.environ['TORCH_CUDA_ARCH_LIST']='.'.join(map(str,torch.cuda.get_device_capability(0)))
run([sys.executable,'-m','pip','install','plyfile==1.1.3','ninja==1.13.0','setuptools==81.0.0','wheel'], 'install_python.log')
DEPS=REPO/'gaussian-splattingWithMask/submodules'
run([sys.executable,'-m','pip','install','--verbose','--no-build-isolation',str(DEPS/'simple-knn'),str(DEPS/'diff-gaussian-rasterization')], 'build_cuda.log')
import simple_knn._C, diff_gaussian_rasterization
run([sys.executable,'check_contract.py','--sparse',INPUT/'sparse/0'], 'check_contract.log')
save('installed_environment.json',dict(pip_freeze=subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True),
    nvidia_smi=subprocess.check_output(['nvidia-smi'],text=True)))
print('CUDA extensions imported successfully.')


$ /usr/bin/python3 -m pip install plyfile==1.1.3 ninja==1.13.0 setuptools==81.0.0 wheel


$ /usr/bin/python3 -m pip install --verbose --no-build-isolation /content/zs601-lidar-init-v001/attempt002/source/gaussian-splattingWithMask/submodules/simple-knn /content/zs601-lidar-init-v001/attempt002/source/gaussian-splattingWithMask/submodules/diff-gaussian-rasterization


Using pip 24.1.2 from /usr/local/lib/python3.13/dist-packages/pip (python 3.13)


Processing /content/zs601-lidar-init-v001/attempt002/source/gaussian-splattingWithMask/submodules/simple-knn


  Preparing metadata (setup.py): started


...

  Stored in directory: /root/.cache/pip/wheels/fd/7b/21/eaa621985b3690f5bf7ed7eeb0b7eb8b6301aebc538f8fc26a


Successfully built simple_knn diff_gaussian_rasterization


$ /usr/bin/python3 check_contract.py --sparse /content/zs601-lidar-init-v001/attempt002/input/sparse/0


{'camera_count': 200, 'max_projection_error_px': 2.0463630789890885e-12, 'uint16_mm_roundtrip': True}


CUDA extensions imported successfully.


In [4]:
# Smoke uses three separated fixed camera IDs; it cannot change initialization settings.
base=[sys.executable,'render_from_sparse_v4.py','--point-cloud',INPUT/'points_3cm.ply','--sparse',INPUT/'sparse/0',
      '--opacity','0.999999','--init-scale-factor','0.5']
smoke=RUN/'smoke'
run(base+['--output',smoke,'--view-ids','3001,3301,3598'], 'smoke_render.log')
verify=[sys.executable,'verify_outputs.py','--point-cloud',INPUT/'points_3cm.ply','--sparse',INPUT/'sparse/0']
run(verify+['--output',smoke,'--expected-views','3'], 'smoke_verify.log')
assert (smoke/'COMPLETE.json').is_file()
print('Three-view smoke, analytic CUDA depth, PNGs, poses and PLY reload passed.')


$ /usr/bin/python3 render_from_sparse_v4.py --point-cloud /content/zs601-lidar-init-v001/attempt002/input/points_3cm.ply --sparse /content/zs601-lidar-init-v001/attempt002/input/sparse/0 --opacity 0.999999 --init-scale-factor 0.5 --output /content/zs601-lidar-init-v001/attempt002/run-001/smoke --view-ids 3001,3301,3598


Number of points at initialisation :  309240


{"index": 1, "image_id": 3001, "name": "003001.png", "visible_gaussians": 59325, "coverage_alpha95": 0.7598375451263538, "coverage_alpha50": 0.995932987364621, "seconds": 1.2882616519927979}


{"index": 3, "image_id": 3598, "name": "003598.png", "visible_gaussians": 140543, "coverage_alpha95": 0.8494373307761733, "coverage_alpha50": 0.9950910988267148, "seconds": 1.3243203163146973}


RENDER_COMPLETE


...

  },


  "evaluation": "Blender synthetic GT, fixed near views; no real-image reconstruction or optimized 3DGS claims",


  "ssim": "RGB [0,1], Gaussian 11x11 sigma1.5, C1=0.01^2 C2=0.03^2, fully valid windows"


}


Three-view smoke, analytic CUDA depth, PNGs, poses and PLY reload passed.


In [5]:
# Formal run: same settings and all 200 cameras; no training calls.
formal=RUN/'formal'
run(base+['--output',formal], 'formal_render.log')
run(verify+['--output',formal,'--expected-views','200','--ground-truth',INPUT/'gt'], 'formal_verify.log')
report=json.loads((formal/'verification.json').read_text())
assert report['views']==200 and report['optimization_steps']==0
assert json.loads((smoke/'initialization.json').read_text())['gaussian_ply_sha256'] == json.loads((formal/'initialization.json').read_text())['gaussian_ply_sha256']
save('RUN_COMPLETE.json',dict(status='VERIFIED_INITIALIZATION_NOT_TRAINED',source_commit=env['source_commit'],
    optimization_steps=0,views=report['views'],points=report['points'],smoke_and_formal_same_ply=True))
print(json.dumps(report,indent=2))


$ /usr/bin/python3 render_from_sparse_v4.py --point-cloud /content/zs601-lidar-init-v001/attempt002/input/points_3cm.ply --sparse /content/zs601-lidar-init-v001/attempt002/input/sparse/0 --opacity 0.999999 --init-scale-factor 0.5 --output /content/zs601-lidar-init-v001/attempt002/run-001/formal


Number of points at initialisation :  309240


{"index": 1, "image_id": 3001, "name": "003001.png", "visible_gaussians": 59325, "coverage_alpha95": 0.7598375451263538, "coverage_alpha50": 0.995932987364621, "seconds": 1.2796070575714111}


{"index": 11, "image_id": 3031, "name": "003031.png", "visible_gaussians": 3306, "coverage_alpha95": 0.32578407039711194, "coverage_alpha50": 0.9869063064079422, "seconds": 0.9113667011260986}


{"index": 21, "image_id": 3061, "name": "003061.png", "visible_gaussians": 100011, "coverage_alpha95": 0.8337390004512636, "coverage_alpha50": 0.9957285085740072, "seconds": 1.323333740234375}


...

  },


  "evaluation": "Blender synthetic GT, fixed near views; no real-image reconstruction or optimized 3DGS claims",


  "ssim": "RGB [0,1], Gaussian 11x11 sigma1.5, C1=0.01^2 C2=0.03^2, fully valid windows"


}


{
  "status": "VERIFIED_INITIALIZATION_NOT_TRAINED",
  "views": 200,
  "points": 309240,
  "optimization_steps": 0,
  "exact_xyz": true,
  "max_sh0_rgb_error": 7.807039748009004e-08,
  "opacity_decoded_min": 0.9999989867206773,
  "opacity_decoded_max": 0.9999989867206773,
  "png_count": 1400,
  "camera_pose_and_intrinsics_exact": true,
  "means_over_views": {
    "alpha95_coverage": 0.6966529853903429,
    "depth_coverage": 0.9926297735221119,
    "native_psnr_gt_valid_db": 21.005499196135425,
    "straight_psnr_gt_valid_db": 23.290285885084234,
    "straight_psnr_covered_db": 22.58943694006038,
    "native_ssim_gt_valid": 0.7454252585768699,
    "straight_ssim_covered": 0.7983650952577591,
    "ssim_covered_window_fraction": 0.4003830451827617,
    "depth_mae_mm": 34.174325324089175,
    "depth_rmse_mm": 95.07913194531957,
    "depth_absrel": 0.01758937660636064,
    "depth_eval_coverage": 0.9335087150270758
  },
  "evaluation": "Blender synthetic GT, fixed near views; no real-image r

## Interpretation

`formal/point_cloud/iteration_0/point_cloud.ply` is the saved Gaussian scene.
`formal/images/` contains 200 native black-background renders. `color/` preserves
straight RGB as in v4; the alpha and masks identify missing cloud coverage.
Metrics compare initialization renderings with synthetic Blender GT only.
They are not a 3DGS training result. The notebook runner saves this actually
executed notebook, including execution counts and outputs, before packaging artifacts.
